<font face="Arial" color="black" size="6">**定义GPT模型**</font>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import inspect
from dataclasses import dataclass


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304  # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embed: int = 768
    dropout: float = 0.0
    bias: bool = True


class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embed % config.n_head == 0
        self.qkv_attn = nn.Linear(config.n_embed, 3 * config.n_embed, bias=config.bias)  # qkv合并为一个大矩阵，方便计算
        self.c_proj = nn.Linear(config.n_embed, config.n_embed, bias=config.bias)  # output projection
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embed = config.n_embed
        # 记得有一篇论文说head_size要等于seq_length才合理
        self.head_size = config.n_embed // config.n_head

    def forward(self, x):
        B, T, C = x.size()  # bach, seq, embed
        q, k, v = self.qkv_attn(x).split(self.n_embed, dim=2)  # qkv大矩阵分为q,k,v3个小矩阵
        q.view(B, T, self.n_head, self.head_size).transpose(1, 2)  # (B, nh, T, hs)
        k.view(B, T, self.n_head, self.head_size).transpose(1, 2)  # (B, nh, T, hs)
        v.view(B, T, self.n_head, self.head_size).transpose(1, 2)  # (B, nh, T, hs)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout if self.training else 0,
                                           is_causal=True)  # is_causal为因果掩码，即当前位置之前的位置不能被访问
        # (B, nh, T, hs) * (B, nh, hs, T) -> (B, nh, T, T)  q * k.T/sqrt(hs)
        # (B, nh, T, T)  * (B, nh, T, hs) -> (B, nh, T, hs)  y = (q * k.T/sqrt(hs)) * v
        y.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, nh, hs) -> (B,T,C)
        y = self.resid_dropout(self.c_proj(y))  # (B,T,C)
        return y


class MLP(nn.Module):
    # MLP部分参考llama MLP结构
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embed, 4 * config.n_embed, bias=config.bias)
        self.c_proj = nn.Linear(4 * config.n_embed, config.n_embed, bias=config.bias)
        self.act = nn.functional.relu
        self.dropout = nn.Dropout(config.dropout)
        self.gate = nn.Linear(config.n_embed, 4 * config.n_embed, bias=config.bias)

    def forward(self, x):
        x = self.c_fc(x)
        gate_proj = self.gate(x)
        # llama中的代码：
        # intermediate_states = (self.act_fn(gate_proj) * up_proj).split(slice, dim=2)
        # nanogpt的
        # x = self.act_func(x)
        # 发现这里区别主要在，nanogpt对upproj的x使用激活函数，llama则是对gate使用
        x = self.act(gate_proj) * x
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln = LayerNorm(config.n_embed, bias=False)  # RMS Norm
        self.attn = CausalSelfAttention(config)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_(x))  # pre-norm
        x = x + self.mlp(self.ln(x))
        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embed),  # token embedding
            wpe=nn.Embedding(config.block_size, config.n_embed),  # position embedding
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln=LayerNorm(config.n_embed, bias=False),
        ))
        self.lm_head = nn.Linear(config.n_embed, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # 共享参数 词表(tokens)->词向量(embedding)/词向量->词表
        num_params = 0
        self.apply(self._init_weights)  # linear 和 embedding 初始化
        for pname, p in self.named_parameters():
            num_params += p.numel()
            if pname.endswith('bias'): # _init_weight中只是先置零，这里精细调整偏置
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
        print(f'模型参数量为：{num_params}')

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):  # targets是训练时传入的目标，用来计算交叉熵loss
        device = idx.device
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=device)
        # embedding
        tok_emb = self.transformer.wte(idx)  # (B,T,n_embed)
        pos_emb = self.transformer.wpe(pos)  # (T,n_embed)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln(x)
        # 经过lm_head
        # target= True 表示模型正在训练阶段，需要回传loss
        # logits取最后一个（-1）即生成出来的东西，这样和目标的一个token维度相同，才好计算损失
        if targets is not None:  # 训练阶段
            logits = self.lm_head(x) # (B,T,vocab_size)
            # (B*T,vocab_size),(B*T,) 忽略标签-1, 计算交叉熵损失
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:  # 预测阶段
            logits = self.lm_head(x)
            loss = None
        return logits, loss

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        param_dict = {pn: p for pn, p in self.named_parameters()}
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # 对二维参数进行权重衰减
        decay_params = [p for pn, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for pn, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"{num_decay_params}个参数使用权重衰减，{num_nodecay_params}个参数不使用权重衰减")
        # 启用fused
        fused_avail = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = (device_type == 'cuda') and fused_avail
        if use_fused:
            print("AdamW optimiser using fused!")
        extra_args = {'fused': True} if use_fused else {}
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        # ** 用于将一个字典解包成关键字参数传递给函数
        return optimizer

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # [B,T,vocab_size] 取最后一个生成的token的logits
            # tempreture更高，生成的随机性更高
            # 从这里能知道，是softmax的性质决定的，指数函数小的时候变化小，不同token的probs差距会被减少，随机性就强了
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')  # 忽略topk名以后的token
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

<font face="Arial" color="black" size="6">**准备数据集**</font>

In [ ]:
import os
import numpy as np
from transformers import AutoTokenizer

# 使用本地Qwen2.5 tokenizer
tokenizer_path = "./tokenizer/Qwen2.5-7B-Instruct"
print(f"正在加载本地tokenizer: {tokenizer_path}")
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_path,
    local_files_only=True
)
print("本地tokenizer加载成功！")

# 读取数据
# input_file_path = os.path.join(os.path.dirname(__file__), 'test.txt')
input_file_path = "./data/vtuber/cww.txt"
with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read() # f.read()会自动将utf-8字节序列转换为Unicode字符串
print(f"原始数据长度: {len(data)} 字符")

# 数据分割
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

# 使用Qwen2.5 tokenizer编码
print("正在编码训练数据...")
train_ids = tokenizer.encode(train_data, add_special_tokens=False)
print("正在编码验证数据...")
val_ids = tokenizer.encode(val_data, add_special_tokens=False)

print(f"train has {len(train_ids):,} tokens")
print(f"val has {len(val_ids):,} tokens")
print(f"训练数据压缩比: {len(train_data)/len(train_ids):.2f} 字符/token")
print(f"验证数据压缩比: {len(val_data)/len(val_ids):.2f} 字符/token")

# 检查token范围
max_token_id = max(max(train_ids), max(val_ids))
print(f"最大token ID: {max_token_id}")

# 使用uint32而不是uint16来避免溢出
print("保存为uint32格式...")
train_ids = np.array(train_ids, dtype=np.uint32)  # 改为uint32
val_ids = np.array(val_ids, dtype=np.uint32)     # 改为uint32

# train_ids.tofile(os.path.join(os.path.dirname(__file__), 'train.bin'))
# val_ids.tofile(os.path.join(os.path.dirname(__file__), 'val.bin'))
train_ids.tofile('./data/vtuber/train.bin')
val_ids.tofile('./data/vtuber/val.bin')
print("数据处理完成！")
print(f"训练数据保存到: train.bin ({len(train_ids)} tokens)")
print(f"验证数据保存到: val.bin ({len(val_ids)} tokens)")

<font face="Arial" color="black" size="6">**模型训练**</font>

In [ ]:
import os
import time
import math
import torch
import numpy as np
import torch.nn as nn
from model import GPTConfig,GPT

# 模型参数
block_size = 128  # 窗口大小GPT2为1024
gradient_accumulation_steps = 5 * 8  # used to simulate larger batch sizes
batch_size = 32  # if gradient_accumulation_steps > 1, this is the micro-batch size
n_layer = 12
n_head = 6
n_embed = 768
bias = False
dropout = 0.0  # for pretraining 0 is good, for finetuning try 0.1+
init_from = 'scratch'  # 'scratch' or 'resume' # 从头训练还是继续
checkpoint_save_dir = 'checkpoints'
eval_iters = 200
eval_interval = 200  # 每n步eval和保存checkpoint一次
always_save_checkpoint = True

# 学习率衰减 - 调整为更合理的值
learning_rate = 3e-4  # 降低学习率
warmup_iters = 200
lr_decay_iters = 2000  # 延长衰减周期
min_lr = 3e-5

# 优化器参数
max_iters = 2000  # 增加训练步数
weight_decay = 1e-1
betas = (0.9, 0.95)
grad_clip = 1.0  # 梯度裁剪

# system
device = 'cuda'
device_type = 'cuda'
compile = True
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
# 在后续上下文中会按照dtype以及具体运算方法自动调整精度 dtype参数类似于基准精度
ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype)

# dataloader
dataset_path = 'data/vtuber'
data_dir = os.path.join(dataset_path)


def get_batch(split):
    if split == 'train':
        data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint32, mode='r')
    else:
        data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint32, mode='r')

    ix = torch.randint(len(data) - block_size, (batch_size,))  # (batch_size,) 随机取batch_size个样本一维索引矩阵
    x = torch.stack(
        [torch.from_numpy((data[i:i + block_size].astype(np.int64))) for i in ix])  # (batch_size, block_size)
    y = torch.stack([torch.from_numpy((data[i + 1:i + 1 + block_size].astype(np.int64))) for i in
                     ix])  # (batch_size, block_size) x向后移一位，作为训练时的预测标签
    # 固定在内存中，直接从内存访问，并不经过cpu，non_blocking表示异步操作，不影响cpu继续运行代码
    print(x.device)
    print(y.device)
    x, y = x.contiguous().pin_memory().to(device, non_blocking=True), y.contiguous().pin_memory().to(device,
                                                                                                     non_blocking=True)
    return x, y


model_args = dict(n_layer=n_layer, n_head=n_head, n_embed=n_embed, block_size=block_size, bias=bias, vocab_size=None,
                  dropout=dropout)

iter_num = 0
best_val_loss = 1e9

assert init_from == 'scratch' or init_from == 'resume'
if init_from == 'scratch':
    print("从头训练模型")
    # 根据prepare.py的输出，最大token ID是151603，所以设置为151604
    model_args['vocab_size'] = 129184  # 修正词汇表大小
    gpt_args = GPTConfig(**model_args)
    model = GPT(gpt_args)

elif init_from == 'resume':
    print("继续训练模型")
    ckpt_path = os.path.join(checkpoint_save_dir, 'checkpoint.pt')
    checkpoint = torch.load(ckpt_path, map_location=device)
    checkpoint_model_args = checkpoint['model_args']
    for k in ['n_layer', 'n_head', 'n_embed', 'block_size', 'bias', 'vocab_size']:
        model_args[k] = checkpoint_model_args[k]
    gpt_args = GPTConfig(**model_args)
    model = GPT(gpt_args)
    state_dict = checkpoint['model']
    model.load_state_dict(state_dict)
    iter_num = checkpoint['iter_num']
    best_val_loss = checkpoint['best_val_loss']

scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))  # 当目前混合精度为float16时启动梯度自动缩放，防止梯度下溢
model.to(device)
optimizer = model.configure_optimizers(weight_decay, learning_rate, betas, device_type)
if init_from == 'resume':
    optimizer.load_state_dict(checkpoint['optimizer'])
checkpoint = None
# compile the model
if compile:
    print("compiling the model... (takes a ~minute)")
    unoptimized_model = model
    model = torch.compile(model)  # requires PyTorch 2.0


# 评估训练集和验证集损失
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


# learning rate decay scheduler (cosine with warmup)
def get_lr(now_iter):
    if (now_iter < warmup_iters):  # lr warmup
        return learning_rate * (now_iter + 1) / (warmup_iters + 1)
    elif (now_iter > lr_decay_iters):  # lr decay
        return min_lr
    else:  # lr cosine decay
        rate = (now_iter - warmup_iters) / (lr_decay_iters - warmup_iters)
        coeff = 0.5 * (1.0 + math.cos(math.pi * rate))  # coeff ranges 0..1
        return min_lr + coeff * (learning_rate - min_lr)  # ranges min_lr..learning_rate 余弦下降


# 创建checkpoint目录
os.makedirs(checkpoint_save_dir, exist_ok=True)

# 训练
t_before = time.time()
X, Y = get_batch('train')
# 初始评估
if iter_num == 0:
    loss_dict = estimate_loss()
    print(f"初始状态 - train_loss: {loss_dict['train']:.4f}, val_loss: {loss_dict['val']:.4f}")
while True:
    # determine and set the learning rate for this iteration
    lr = get_lr(iter_num)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    # evaluate the loss on train/val sets and write checkpoints
    if iter_num > 0 and iter_num % eval_interval == 0:
        loss_dict = estimate_loss()
        print(f"iter {iter_num} - train_loss: {loss_dict['train']:.4f}, val_loss: {loss_dict['val']:.4f}, lr: {lr:.2e}")
        checkpoint = {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'model_args': model_args,
            'iter_num': iter_num,
            'best_val_loss': best_val_loss
        }
        checkpoint_path = os.path.join(checkpoint_save_dir, f'checkpoint_{iter_num}.pt')
        if loss_dict['val'] < best_val_loss:
            best_val_loss = loss_dict['val']
            print(print(f"新的最佳val loss: {best_val_loss:.4f}"))
            print(f"saving checkpoint to {checkpoint_save_dir}")
            torch.save(checkpoint, checkpoint_path)
        if always_save_checkpoint:
            print(f"saving checkpoint to {checkpoint_save_dir}")
            torch.save(checkpoint, checkpoint_path)
    # forward backward update, with optional gradient accumulation to simulate larger batch size
    # and using the GradScaler if data type is float16
    for micro_step in range(gradient_accumulation_steps):
        X, Y = get_batch('train')
        with ctx:
            logits, loss = model(X, Y)
            # backward pass, with gradient scaling if training in fp16
            loss = loss / gradient_accumulation_steps  # scale the loss to account for gradient accumulation
        scaler.scale(loss).backward()
    if iter_num % 50 == 0:  # 每50步打印一次，减少输出
        print(f"iter: {iter_num}, loss: {loss.item():.4f}, lr: {lr:.2e}")
    # clip the gradient
    if grad_clip > 0.0:
        scaler.unscale_(optimizer)  # 损失使用了缩放，梯度反缩放才能进行裁剪
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)  # 使用缩放的梯度执行优化器步骤
    scaler.update()  # 根据梯度的溢出情况动态调整缩放比例
    optimizer.zero_grad(set_to_none=True)  # 清空梯度缓存，为下一次迭代做准

    t_after = time.time()
    dt = t_after - t_before
    t_before = t_after

    iter_num += 1
    if iter_num > max_iters:
        print(f"训练完成！总共训练了{max_iters}步")
        break
# 最终评估
print("进行最终评估...")
final_losses = estimate_loss()
print(f"最终结果 - train_loss: {final_losses['train']:.4f}, val_loss: {final_losses['val']:.4f}")

<font face="Arial" color="black" size="6">**测试样例**</font>

In [ ]:
import os
import torch
from Re_Zero_LLM.model import GPT, GPTConfig
from transformers import AutoTokenizer
# 配置参数
checkpoint_save_dir = 'checkpoints'
device = 'cuda'
device_type = 'cuda'
dtype = 'bfloat16'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

# 生成参数
top_k = 50
max_new_tokens = 50

print("加载模型...")

# 加载checkpoint
ckpt_path = os.path.join(checkpoint_save_dir, 'checkpoint.pt')
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

# 初始化模型
args = checkpoint['model_args']
model = GPT(GPTConfig(**args))

# 加载权重
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.eval()
model.to(device)

# 加载tokenizer
tokenizer_path = "tokenizer/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, local_files_only=True)

print("模型加载完成！")
def test_generation(prompt):
    """测试单句生成"""
    print(f"\n输入: {prompt}")

    # 编码
    start_ids = tokenizer.encode(prompt, add_special_tokens=False)
    x = torch.tensor(start_ids, dtype=torch.long, device=device).unsqueeze(0)

    # 生成
    ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype)
    with torch.no_grad():
        with ctx:
            y = model.generate(x, max_new_tokens, top_k=top_k)
            generated_text = tokenizer.decode(y[0].tolist())

    print(f"输出: {generated_text}")
    return generated_text
# 单句测试
test_generation("我们可以做朋友嘛？")

# 或者交互式测试
while True:
    user_input = input("\n请输入测试文本 (输入'quit'退出): ")
    if user_input.lower() == 'quit':
        break
    test_generation(user_input)